# Titanic Survival Prediction Project 🚢

## Introduction Section
Welcome to the **Titanic Survival Prediction Project**. This project involves building an end-to-end Machine Learning classification pipeline.

### Overview of Titanic Dataset
The sinking of the Titanic is one of the most infamous shipwrecks in history. The dataset provides information on the fate of passengers, including their age, sex, passenger class, ticket fare, and family size.

### Importance of Classification Models
Classification algorithms learn patterns from historical data and use them to predict discrete categories for new observations. In our case, the model will classify whether a passenger *Survived (1)* or *Did Not Survive (0)*.

### Business & Real-World Relevance
Predictive analytics like this form the core of many modern business applications:
- **Healthcare**: Predicting patient outcomes or disease risks based on demographics and symptoms.
- **Finance**: Predicting credit defaults.
- **Marketing**: Predicting customer churn based on behavior profiles.


## 1. Data Loading & Initial Exploration
Let's load the dataset and get a fundamental understanding of our data structure.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('../data/titanic.csv')
display(df.head())


In [ ]:
print(f"Dataset Shape: {df.shape}")
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

# Class Balance Visualization
plt.figure(figsize=(6, 4))
sns.countplot(x='Survived', data=df, palette='Set2')
plt.title('Class Balance: Survival Count')
plt.show()


## 2. Exploratory Data Analysis (EDA)
Visual storytelling to uncover hidden patterns and understand the influence of passenger attributes on survival.


In [ ]:
# Gender vs Survival
plt.figure(figsize=(6, 4))
sns.countplot(x='Sex', hue='Survived', data=df, palette='Set1')
plt.title('Gender vs Survival (Women had higher survival rates)')
plt.show()

# Pclass vs Survival
plt.figure(figsize=(6, 4))
sns.countplot(x='Pclass', hue='Survived', data=df, palette='viridis')
plt.title('Passenger Class vs Survival (1st class had higher survival rates)')
plt.show()


In [ ]:
# Age Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df[df['Survived']==0]['Age'].dropna(), kde=True, color='red', label='Died', bins=30)
sns.histplot(df[df['Survived']==1]['Age'].dropna(), kde=True, color='green', label='Survived', bins=30)
plt.title('Age Distribution by Survival')
plt.legend()
plt.show()


In [ ]:
# Fare Distribution (Outliers Analysis)
plt.figure(figsize=(8, 5))
sns.boxplot(x='Survived', y='Fare', data=df, palette='Set3')
plt.title('Fare Boxplot (Higher fares correlate with survival)')
plt.ylim(0, 200) # Zoomed in for clarity
plt.show()


## 3. Advanced Feature Engineering
We will engineer new features to help the model find better patterns.
1. **Title Extraction**: Extracting titles (Mr, Mrs, Miss, Master, etc.) from the 'Name' column gives clues about social status and age.
2. **Family Size**: `SibSp` (siblings/spouses) + `Parch` (parents/children) + 1 (self).
3. **IsAlone**: A binary flag indicating if a passenger was traveling alone.
4. **HasCabin**: A binary flag indicating if the passenger had a recorded cabin number.


In [ ]:
# 1. Title Extraction
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
rare_titles = ['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# 2. Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 3. IsAlone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 4. Cabin Presence
df['HasCabin'] = df['Cabin'].notnull().astype(int)

# Drop unneeded text columns
df.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'], inplace=True)
display(df.head())


## 4. Missing Value Handling & Preprocessing Pipeline
We use `scikit-learn` Pipelines to prevent data leakage and streamline operations.
- **Numerical (Age, Fare)**: Imputed using median strategy, then `StandardScaler`.
- **Categorical (Sex, Pclass, Embarked, Title)**: Imputed using most frequent, then `OneHotEncoder`.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Note: The target has a few NaNs in the full OpenML dataset (since it includes test set)
# Let's drop rows where Survived is NaN for training
df = df.dropna(subset=['Survived'])
df['Survived'] = df['Survived'].astype(int)

X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title']
passthrough_features = ['IsAlone', 'HasCabin']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
        ('pass', 'passthrough', passthrough_features)
    ])


## 5. Model Training & Hyperparameter Tuning
We train Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting. We tune the ensembles using `GridSearchCV`.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Logistic Regression
lr_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', LogisticRegression(random_state=42))])
lr_pipe.fit(X_train, y_train)

# Random Forest Tuning
rf_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', RandomForestClassifier(random_state=42))])
rf_grid = GridSearchCV(rf_pipe, {'model__n_estimators': [100, 200], 'model__max_depth': [5, 10]}, cv=3, scoring='roc_auc')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

# Gradient Boosting Tuning
gb_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', GradientBoostingClassifier(random_state=42))])
gb_grid = GridSearchCV(gb_pipe, {'model__n_estimators': [100, 200], 'model__learning_rate': [0.05, 0.1]}, cv=3, scoring='roc_auc')
gb_grid.fit(X_train, y_train)
best_gb = gb_grid.best_estimator_

models = {
    'Logistic Regression': lr_pipe,
    'Decision Tree': Pipeline(steps=[('preprocessor', preprocessor), ('model', DecisionTreeClassifier(random_state=42))]).fit(X_train, y_train),
    'Random Forest': best_rf,
    'Gradient Boosting': best_gb
}
print("Models trained successfully!")


## 6. Evaluation Metrics
We evaluate using Accuracy, F1-Score, ROC-AUC, and Confusion Matrices.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_pred_proba)
    })
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
    plt.title(f'Confusion Matrix: {name}')
    plt.show()

results_df = pd.DataFrame(results)
display(results_df)

# Model Comparison
results_df.set_index('Model')[['Accuracy', 'ROC_AUC']].plot(kind='bar', figsize=(10, 6))
plt.title('Accuracy and ROC-AUC Comparison')
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.show()


## 7. Explainable AI (SHAP)
Explainable AI (XAI) bridges the gap between complex model outputs and human understanding. Using **SHAP** (SHapley Additive exPlanations), we can see exactly which features drove the model's predictions.

- **Gender (Sex_male)**: Strongly negatively impacted survival.
- **Pclass**: Lower classes (higher numbers) negatively impacted survival.
- **Age**: Younger age (children) positively impacted survival.


In [ ]:
import shap

# Transform training data to feed into SHAP
X_train_trans = best_gb.named_steps['preprocessor'].transform(X_train)

# Extract feature names
cat_encoder = best_gb.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_feats = list(cat_encoder.get_feature_names_out(categorical_features))
all_feats = numeric_features + cat_feats + passthrough_features

# Generate SHAP values
explainer = shap.TreeExplainer(best_gb.named_steps['model'])
shap_values = explainer.shap_values(X_train_trans)

# Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train_trans, feature_names=all_feats, show=False)
plt.title('SHAP Feature Importance (Gradient Boosting)')
plt.show()


## 8. Conclusion & Future Scope
- **Insights**: The model validates the historical "Women and children first" protocol. Gender, Passenger Class, and Title were the most vital predictors.
- **Model Performance**: Gradient Boosting provided the highest ROC-AUC and stability, effectively handling the mixture of continuous and categorical features.
- **Future Scope**: Experimenting with deep learning algorithms or deploying this notebook via an interactive web application (Streamlit/FastAPI) to predict survival probability for custom inputs.
